### Importación de librerías

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.types import NumericType, StringType
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline
import os

### Conexión

In [2]:
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] += os.pathsep + r"C:\hadoop\bin"

In [3]:
spark = (SparkSession.builder.appName("Proyecto_PySpark").master("local[*]").getOrCreate())

/usr/local/lib/python3.11/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/26 03:20:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/26 03:20:57 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


### Importación de datos.

In [4]:
df_2025 = spark.read.parquet("df_2025.parquet")
df_2026 = spark.read.parquet("df_2026.parquet")

### Variables utilizadas

In [5]:
variables_numericas = ["edad", "antiguedad_total", "horas_semanales"]
variables_categoricas = ["nivel_educativo", "categoria_ocupacional", "dominio"]
variable_objetivo = "salario_mensual"

### Separación validación y entrenamiento

In [6]:
train_df, valid_df = df_2025.randomSplit(weights=[0.75, 0.25], seed=0)

## Inciso 5

#### Variables categóricas

In [7]:
indexers = [StringIndexer(inputCol=columna,
                          outputCol=f"{columna}_index",
                          handleInvalid="keep")
            for columna in variables_categoricas]

In [8]:
encoder = OneHotEncoder(inputCols=[f"{columna}_index" for columna in variables_categoricas],
                        outputCols=[f"{columna}_ohe" for columna in variables_categoricas],
                        dropLast=True)

#### Vector Assembler

In [9]:
assembler = VectorAssembler(inputCols=variables_numericas + [f"{columna}_ohe" for columna in variables_categoricas],
                            outputCol="features_sin_escalar")

#### Variables numéricas

In [10]:
scaler = StandardScaler(inputCol="features_sin_escalar", outputCol="features",
                        withMean=True, withStd=True)

#### Evaluadores

In [16]:
evaluator_mae = RegressionEvaluator(labelCol="salario_mensual", predictionCol="prediction", metricName="mae")
evaluator_rmse = RegressionEvaluator(labelCol="salario_mensual", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="salario_mensual", predictionCol="prediction", metricName="r2")

### Configuración 1

#### Regresión lineal

In [11]:
lr1 = LinearRegression(featuresCol="features", labelCol=variable_objetivo,
                       predictionCol="prediction", regParam=0.0,
                       elasticNetParam=0.0, maxIter=100)

#### Pipeline

In [12]:
pipeline_lr1 = Pipeline(stages=[*indexers, encoder, assembler, scaler, lr1])

#### Entrenamiento del modelo

In [13]:
modelo_lr1 = pipeline_lr1.fit(train_df)

26/09/26 03:17:11 WARN Instrumentation: [36e7bddf] regParam is zero, which might cause numerical instability and overfitting.
26/09/26 03:17:13 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/09/26 03:17:13 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
26/09/26 03:17:14 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
26/09/26 03:17:14 WARN Instrumentation: [36e7bddf] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
                                                                                

#### Predicción de validación

In [14]:
pred_1 = modelo_lr1.transform(valid_df)

#### Métricas

In [15]:
mae_1 = evaluator_mae.evaluate(pred_1)
rmse_1 = evaluator_rmse.evaluate(pred_1)
r2_1 = evaluator_r2.evaluate(pred_1)

print("Configuración 1")
print("MAE:", mae_1)
print("RMSE:", rmse_1)
print("R²:", r2_1)

[Stage 22:=>                                                      (1 + 31) / 32]

Configuración 1
MAE: 1252.8545466613405
RMSE: 2378.2673323313525
R²: 0.3875454955793092


### Configuración 2

#### Regresión lineal

In [16]:
lr2 = LinearRegression(featuresCol="features", labelCol="salario_mensual",
                        predictionCol="prediction", regParam=1,
                        elasticNetParam=0.4, maxIter=100)

#### Pipeline

In [17]:
pipeline_lr2 = Pipeline(stages=[*indexers, encoder, assembler, scaler, lr2])

#### Entrenamiento del modelo

In [18]:
modelo_lr2 = pipeline_lr2.fit(train_df)

#### Predicción de validación

In [19]:
pred_2 = modelo_lr2.transform(valid_df)

#### Métricas

In [20]:
mae_2 = evaluator_mae.evaluate(pred_2)
rmse_2 = evaluator_rmse.evaluate(pred_2)
r2_2 = evaluator_r2.evaluate(pred_2)

print("Configuración 2")
print("MAE:", mae_2)
print("RMSE:", rmse_2)
print("R²:", r2_2)

[Stage 44:==================================>                    (20 + 12) / 32]

Configuración 2
MAE: 1252.6897914983779
RMSE: 2378.2933426461773
R²: 0.3875320990856763


## Inciso 6

### Configuración 1

#### Random Forest

In [21]:
rf_1 = RandomForestRegressor(featuresCol="features", labelCol="salario_mensual",
                             predictionCol="prediction", numTrees=75,
                             maxDepth=5, seed=0)

#### Pipeline

In [22]:
pipeline_rf_1 = Pipeline(stages=[*indexers, encoder, assembler, scaler, rf_1])

#### Entrenamiento del modelo

In [23]:
modelo_rf_1 = pipeline_rf_1.fit(train_df)

#### Predicción de validación

In [24]:
pred_rf_1 = modelo_rf_1.transform(valid_df)

#### Métricas

In [25]:
mae_rf_1 = evaluator_mae.evaluate(pred_rf_1)
rmse_rf_1 = evaluator_rmse.evaluate(pred_rf_1)
r2_rf_1 = evaluator_r2.evaluate(pred_rf_1)

print("Configuración 1")
print("MAE:", mae_rf_1)
print("RMSE:", rmse_rf_1)
print("R²:", r2_rf_1)

[Stage 76:=>                                                      (1 + 31) / 32]

Configuración 1
MAE: 1190.533691607997
RMSE: 2340.670143367945
R²: 0.4067565892877677


### Configuración 2

#### Random Forest

In [11]:
rf_2 = RandomForestRegressor(featuresCol="features", labelCol="salario_mensual",
                             predictionCol="prediction", numTrees=100,
                             maxDepth=7, seed=0)

#### Pipeline

In [12]:
pipeline_rf_2 = Pipeline(stages=[*indexers, encoder, assembler, scaler, rf_2])

#### Entrenamiento del modelo

In [13]:
modelo_rf_2 = pipeline_rf_2.fit(train_df)

26/09/26 03:22:02 WARN DAGScheduler: Broadcasting large task binary with size 1049.0 KiB
26/09/26 03:22:02 WARN DAGScheduler: Broadcasting large task binary with size 1841.2 KiB


#### Predicción de validación

In [14]:
pred_rf_2 = modelo_rf_2.transform(valid_df)

#### Métricas

In [17]:
mae_rf_2 = evaluator_mae.evaluate(pred_rf_2)
rmse_rf_2 = evaluator_rmse.evaluate(pred_rf_2)
r2_rf_2 = evaluator_r2.evaluate(pred_rf_2)

print("Configuración 2")
print("MAE:", mae_rf_2)
print("RMSE:", rmse_rf_2)
print("R²:", r2_rf_2)

[Stage 36:=>                                                      (1 + 31) / 32]

Configuración 2
MAE: 1133.5899736800457
RMSE: 2222.5066594949576
R²: 0.4651418300220984


## Inciso 7